# CIS 5450 Final Project: EV Energy and Thermal Monitoring
*Dee Xie, Hari Raman, Sumit Kolpe, Aditya Bajoria, Aditya Muppavarapu*

## Setup

In [3]:
import importlib.util
import json
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

if importlib.util.find_spec("duckdb") is None:   # not preinstalled in Colab
    %pip install -q duckdb
if importlib.util.find_spec("xgboost") is None:
    %pip install -q xgboost
if importlib.util.find_spec("tqdm") is None:
    %pip install -q tqdm

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

threadpool_limits(limits=1, user_api="blas")   # small matrices: multi-threaded BLAS is slower here (GMM was ~25x slower on Windows)

# Part 1: Data Loading and Cleaning
Turns 98 million raw sensor readings into two clean tables:
- **`tel`**: one row per car per 10 seconds
- **`trips`**: one row per trip

The first run downloads the data (~620 MB) and takes a few minutes. The result is saved to `data/processed/`, so later runs load it in seconds. Set `REBUILD_DATA = True` to rebuild it.

## 1.1 Download the raw data
- Downloads the [TUM EV dataset](https://github.com/TUMFTM/electric-vehicle-uds-dataset) from its [Zenodo archive](https://zenodo.org/records/17532823) on the first run only
- Keeps one file per car (`CUP1`-`CUP5` are CUPRA Born, `ID1`-`ID2` are VW ID.3) and `value_overview.csv`, the list of sensors
- Files go to `data/raw/`; set the `EV_RAW_DIR` environment variable to store them elsewhere

In [4]:
REBUILD_DATA = False   # True = rebuild data/processed even if it already exists
RAW_DIR = Path(os.environ.get("EV_RAW_DIR", "data/raw"))
PROCESSED_DIR = Path("data/processed")
ZENODO_URL = "https://zenodo.org/records/17532823/files/electric-vehicle-uds-dataset-public.zip?download=1"
VEHICLES = ["CUP1", "CUP2", "CUP3", "CUP4", "CUP5", "ID1", "ID2"]
NEEDED_FILES = ["value_overview.csv"] + [f"{v}.parquet" for v in VEHICLES]


def download_raw_data():
    """
    Download the dataset zip and keep only the vehicle Parquet files and the signal list.
    """
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = RAW_DIR / "ev_dataset.zip.part"
    with tqdm(desc="Downloading", unit="B", unit_scale=True, unit_divisor=1024) as bar:
        def show_progress(blocks_done, block_size, total_size):
            done = blocks_done * block_size
            if total_size > 0:
                bar.total, done = total_size, min(done, total_size)
            bar.update(done - bar.n)

        urllib.request.urlretrieve(ZENODO_URL, zip_path, reporthook=show_progress)

    with zipfile.ZipFile(zip_path) as z:
        files = [name for name in z.namelist() if Path(name).name in NEEDED_FILES and not name.startswith("__MACOSX")]
        for name in tqdm(files, desc="Extracting"):
            with z.open(name) as src, open(RAW_DIR / Path(name).name, "wb") as dst:
                shutil.copyfileobj(src, dst)
    zip_path.unlink()


if not all((RAW_DIR / f).exists() for f in NEEDED_FILES):
    download_raw_data()

missing_files = [f for f in NEEDED_FILES if not (RAW_DIR / f).exists()]
assert not missing_files, f"Download incomplete, missing: {missing_files}"
sorted(p.name for p in RAW_DIR.iterdir())

['CUP1.parquet',
 'CUP2.parquet',
 'CUP3.parquet',
 'CUP4.parquet',
 'CUP5.parquet',
 'ID1.parquet',
 'ID2.parquet',
 'value_overview.csv']

## 1.2 Signals we use
- Each reading has a `value_id` that identifies the sensor
- 15 of the dataset's 29 sensors contain data; we keep all 15 and give them readable names
- `bat_temp` is the temperature of the hottest battery cell; `coolant_temp` is the inverter's coolant, not the battery's

In [5]:
SIGNALS = {
    4: "speed",
    900: "soc",
    1200: "hv_voltage",
    56: "hv_aux_power",
    15: "ambient_temp",
    43: "interior_temp",
    1209: "bat_temp",
    1208: "bat_temp_min",
    1272: "bat_inlet_temp",
    1273: "bat_outlet_temp",
    1269: "coolant_temp",
    961: "motor_temp",
    1265: "rotor_temp",
    1205: "ptc_current",
    1207: "ptc_voltage",
}
SIGNAL_COLUMNS = list(SIGNALS.values())

overview = pd.read_csv(RAW_DIR / "value_overview.csv").drop_duplicates("value_id").set_index("value_id")
signal_table = overview.loc[list(SIGNALS), ["name_en", "unit"]]
signal_table.insert(0, "column", SIGNAL_COLUMNS)
signal_table

,column,name_en,unit
value_id,,,
4,speed,vehicle speed,km/h
900,soc,SOC,%
1200,hv_voltage,pack voltage,V
56,hv_aux_power,Power HV Aux,W
15,ambient_temp,Ambient air temperature,°C
43,interior_temp,Interior Temperature,°C
1209,bat_temp,HV Battery Temp max.,°C
1208,bat_temp_min,HV Battery Temp min.,°C
1272,bat_inlet_temp,Battery pack inlet temp,°C


## 1.3 The raw data
- One row per sensor reading: car, time, sensor ID and value
- Some timestamps are impossible (e.g. year 2087) due to logger clock errors, so we keep only 2020-2024

In [6]:
RAW = f"read_parquet({[(RAW_DIR / f'{v}.parquet').as_posix() for v in VEHICLES]})"
VALID_TIME = "CAST(time AS TIMESTAMP) BETWEEN TIMESTAMP '2020-01-01' AND TIMESTAMP '2024-12-31'"

display(duckdb.sql(f"SELECT * FROM {RAW} LIMIT 5").df())

raw_summary = duckdb.sql(f"""
    SELECT vehicle_id,
           COUNT(*) AS readings,
           COUNT(*) FILTER (WHERE NOT ({VALID_TIME})) AS impossible_timestamps,
           MIN(time) FILTER (WHERE {VALID_TIME})::DATE AS first_day,
           MAX(time) FILTER (WHERE {VALID_TIME})::DATE AS last_day
    FROM {RAW}
    GROUP BY vehicle_id
    ORDER BY vehicle_id
""").df()
print(f"Raw readings: {raw_summary['readings'].sum():,}")
raw_summary

,vehicle_id,time,value_id,value
0,CUP1,2087-03-07 15:45:43.011,4,4.68
1,CUP1,2087-03-07 15:45:43.163,1200,402.50
2,CUP1,2087-03-07 15:45:43.211,1200,402.50
3,CUP1,2087-03-07 15:45:43.229,4,4.65
4,CUP1,2087-03-07 15:45:43.374,1200,402.50


Raw readings: 98,061,214


,vehicle_id,readings,impossible_timestamps,first_day,last_day
0,CUP1,8659492,812,2022-11-15,2023-04-24
1,CUP2,3743488,0,2022-11-21,2023-05-04
2,CUP3,986864,0,2022-11-17,2023-04-23
3,CUP4,4943471,0,2022-11-15,2023-04-14
4,CUP5,3305981,0,2022-11-10,2023-04-23
5,ID1,40401700,0,2021-02-01,2023-06-13
6,ID2,36020218,0,2021-07-26,2023-10-10


## 1.4 Pivot into 10-second rows
- Reshapes the data into one row per car per 10 seconds, with one column per sensor
- Readings within each 10-second window are averaged
- 10 seconds lines up fast sensors (speed, 5 times per second) with slow ones (temperatures, every ~10 seconds)

In [7]:
def load_10s_grid():
    """
    One row per vehicle per 10 seconds; each signal is the mean of its readings in that window.
    """
    columns = ",\n".join(f"AVG(value) FILTER (WHERE value_id = {vid}) AS {name}" for vid, name in SIGNALS.items())
    return duckdb.sql(f"""
        SELECT vehicle_id,
               time_bucket(INTERVAL '10 seconds', CAST(time AS TIMESTAMP)) AS ts,
               {columns}
        FROM {RAW}
        WHERE {VALID_TIME}
          AND value_id IN ({", ".join(str(v) for v in SIGNALS)})
          AND value IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1, 2
    """).df()

## 1.5 Remove invalid values
- Values outside a sensor's physical range are set to missing (e.g. speed above 200 km/h, or the error code 1023.98)
- Exact zeros in battery %, voltage and battery temperatures are logger placeholders, so they are set to missing too

In [8]:
VALID_RANGES = {
    "speed": (0, 200),
    "soc": (0, 100),
    "hv_voltage": (200, 500),
    "hv_aux_power": (-20000, 20000),
    "ambient_temp": (-40, 60),
    "interior_temp": (-40, 80),
    "bat_temp": (-40, 80),
    "bat_temp_min": (-40, 80),
    "bat_inlet_temp": (-40, 80),
    "bat_outlet_temp": (-40, 80),
    "coolant_temp": (-40, 120),
    "motor_temp": (-40, 200),
    "rotor_temp": (-40, 250),
}
ZERO_IS_MISSING = ["soc", "hv_voltage", "bat_temp", "bat_temp_min", "bat_inlet_temp", "bat_outlet_temp"]


def remove_invalid_values(df):
    """
    Returns the cleaned data and, per signal, how many values were removed.
    """
    df = df.copy()
    report = []
    for col, (low, high) in VALID_RANGES.items():
        n_values = df[col].notna().sum()
        zeros = (df[col] == 0) if col in ZERO_IS_MISSING else pd.Series(False, index=df.index)
        df.loc[zeros, col] = np.nan
        out_of_range = df[col].notna() & ~df[col].between(low, high)
        df.loc[out_of_range, col] = np.nan
        report.append({"column": col, "values": n_values, "placeholder_zeros": zeros.sum(),
                       "out_of_range": out_of_range.sum()})
    return df, pd.DataFrame(report)

## 1.6 Remove spikes
- Removes one-reading glitches, where a temperature jumps and immediately comes back
- A reading more than 4-8 °C away from the median of its neighbours (70 seconds) is set to missing

In [9]:
SPIKE_LIMITS = {"bat_inlet_temp": 4, "bat_outlet_temp": 4, "coolant_temp": 4, "motor_temp": 6, "rotor_temp": 8,
                "ambient_temp": 4}


def remove_spikes(df):
    """
    Returns the cleaned data and the number of spikes removed per signal.
    """
    df = df.sort_values(["vehicle_id", "ts"]).reset_index(drop=True)
    removed = {}
    for col, limit in SPIKE_LIMITS.items():
        median = df.groupby("vehicle_id")[col].transform(lambda s: s.rolling(7, center=True, min_periods=3).median())
        spikes = (df[col] - median).abs() > limit
        removed[col] = int(spikes.sum())
        df.loc[spikes, col] = np.nan
    return df, removed

## 1.7 Fill short gaps
- Battery temperature is logged every few minutes, so it is interpolated across gaps of up to 10 minutes
- Other sensors repeat their last reading for up to 60 seconds
- Speed is never filled, since that would invent distance
- `bat_temp_observed` marks real (not interpolated) battery temperature readings; forecasting models (RQ2, RQ4) should only use those, because interpolated values are based on future readings

In [10]:
INTERPOLATED = ["bat_temp", "bat_temp_min"]


def interpolate_short_gaps(df, col, max_gap_s=600):
    """
    Time interpolation between two real readings of the same vehicle that are at most max_gap_s apart.
    """
    filled = []
    for _, g in df.groupby("vehicle_id"):
        s = g.set_index("ts")[col]
        seen = s.index.to_series().where(s.notna())
        gap = (seen.bfill() - seen.ffill()).dt.total_seconds()
        filled.append(s.interpolate(method="time", limit_area="inside").where(gap <= max_gap_s).to_numpy())
    return np.concatenate(filled)


def forward_fill_short_gaps(df, col, max_gap_s=60):
    """
    Carry the last reading forward, but only while it is at most max_gap_s old.
    """
    last_seen = df["ts"].where(df[col].notna()).groupby(df["vehicle_id"]).ffill()
    filled = df[col].groupby(df["vehicle_id"]).ffill()
    return filled.where((df["ts"] - last_seen).dt.total_seconds() <= max_gap_s)


def fill_missing(df):
    """
    Expects rows sorted by vehicle and time (as returned by remove_spikes).
    """
    df = df.copy()
    for col in INTERPOLATED:
        df[f"{col}_observed"] = df[col].notna()
        df[col] = interpolate_short_gaps(df, col)
    for col in SIGNAL_COLUMNS:
        if col not in ["speed", *INTERPOLATED]:
            df[col] = forward_fill_short_gaps(df, col)
    return df

## 1.8 Distance and acceleration
- Distance per row = speed × 10 seconds, since there is no odometer
- Acceleration = change in speed between rows, left empty if rows are more than 15 seconds apart

In [11]:
def add_motion(df):
    df = df.copy()
    df["dt_s"] = df.groupby("vehicle_id")["ts"].diff().dt.total_seconds()
    df["dist_km"] = df["speed"].fillna(0) * 10 / 3600
    speed_change = df.groupby("vehicle_id")["speed"].diff() / 3.6
    df["accel_ms2"] = (speed_change / df["dt_s"]).where(df["dt_s"] <= 15)
    return df

## 1.9 Split into trips
- A car is moving when its speed is above 2 km/h
- Stops shorter than 5 minutes stay inside the trip
- Trips shorter than 2 minutes or 0.5 km are dropped
- Adds `trip_id`, time and distance since the trip started, and `state` (driving or parked) to every row

In [12]:
def split_trips(df, moving_kmh=2, max_stop_s=300, min_s=120, min_km=0.5):
    df = df.copy()
    df["trip_id"] = None
    for vehicle, g in df.groupby("vehicle_id"):
        moving_ts = g.loc[g["speed"] > moving_kmh, "ts"]
        trip_no = moving_ts.diff().dt.total_seconds().gt(max_stop_s).cumsum()
        starts = moving_ts.groupby(trip_no).min().to_numpy()
        ends = moving_ts.groupby(trip_no).max().to_numpy()

        idx = np.searchsorted(starts, g["ts"].to_numpy(), side="right") - 1
        in_trip = (idx >= 0) & (g["ts"].to_numpy() <= ends[np.clip(idx, 0, None)])
        raw_trip = pd.Series(np.where(in_trip, idx, -1), index=g.index)

        duration = (ends - starts) / np.timedelta64(1, "s") + 10
        distance = g["dist_km"].groupby(raw_trip).sum()
        keep = [k for k in range(len(starts)) if duration[k] >= min_s and distance.get(k, 0) >= min_km]
        names = {k: f"{vehicle}_{i + 1:04d}" for i, k in enumerate(keep)}
        df.loc[g.index, "trip_id"] = raw_trip.map(names)

    in_trip = df["trip_id"].notna()
    grouped = df[in_trip].groupby("trip_id")
    df.loc[in_trip, "trip_elapsed_s"] = (df.loc[in_trip, "ts"] - grouped["ts"].transform("min")).dt.total_seconds()
    df.loc[in_trip, "trip_dist_km"] = grouped["dist_km"].cumsum()
    df["state"] = np.where(in_trip, "driving", "parked")
    return df

## 1.10 Trip table
- One row per trip: driving style (speed, stops, acceleration), temperatures and energy use
- Energy use (`soc_per_100km`) is the % of battery used per 100 km
- Battery % is logged in 0.4% steps, so energy is unreliable on short trips; `energy_valid_5km` and `energy_valid_20km` mark trips of at least 5 or 20 km with a believable value (5-100 % per 100 km)

In [13]:
def build_trip_table(tel):
    t = tel[tel["trip_id"].notna()].copy()
    t["stopped"] = t["speed"] <= 2
    g = t.groupby("trip_id")

    trips = pd.DataFrame({
        "vehicle_id": g["vehicle_id"].first(),
        "start_ts": g["ts"].min(),
        "end_ts": g["ts"].max(),
        "n_rows": g.size(),
        "distance_km": g["dist_km"].sum(),
        "speed_mean_moving": t[~t["stopped"]].groupby("trip_id")["speed"].mean(),
        "speed_mean": g["speed"].mean(),
        "speed_max": g["speed"].max(),
        "speed_std": g["speed"].std(),
        "stop_fraction": g["stopped"].mean(),
        "accel_abs_mean": g["accel_ms2"].apply(lambda s: s.abs().mean()),
        "accel_p95": g["accel_ms2"].quantile(0.95),
        "decel_p05": g["accel_ms2"].quantile(0.05),
        "soc_start": g["soc"].first(),
        "soc_end": g["soc"].last(),
    })
    trips["duration_min"] = (trips["end_ts"] - trips["start_ts"]).dt.total_seconds() / 60 + 10 / 60
    trips["soc_used"] = trips["soc_start"] - trips["soc_end"]
    trips["soc_per_100km"] = 100 * trips["soc_used"] / trips["distance_km"]
    for col in ["bat_temp", "bat_outlet_temp"]:
        trips[f"{col}_start"] = g[col].first()
        trips[f"{col}_end"] = g[col].last()
        trips[f"{col}_max"] = g[col].max()
        trips[f"{col}_rise"] = trips[f"{col}_end"] - trips[f"{col}_start"]
    for col in ["ambient_temp", "hv_voltage", "hv_aux_power", "coolant_temp", "motor_temp", "interior_temp",
                "bat_inlet_temp"]:
        trips[f"{col}_mean"] = g[col].mean()
    trips["start_hour"] = trips["start_ts"].dt.hour
    trips["start_month"] = trips["start_ts"].dt.month
    trips["weekday"] = trips["start_ts"].dt.dayofweek

    believable = (trips["soc_used"] > 0) & trips["soc_per_100km"].between(5, 100)
    trips["energy_valid_5km"] = believable & (trips["distance_km"] >= 5)
    trips["energy_valid_20km"] = believable & (trips["distance_km"] >= 20)
    return trips.reset_index()

## 1.11 Run the pipeline
- Runs steps 1.4-1.10 and saves the results to `data/processed/` (not committed to git)
- If the files already exist, it loads them instead of rebuilding

In [14]:
OUTPUTS = ["telemetry_10s.parquet", "trips.parquet", "trips.csv", "data_quality_report.csv", "pipeline_params.json"]

if REBUILD_DATA or not all((PROCESSED_DIR / f).exists() for f in OUTPUTS):
    tel = load_10s_grid()
    missing_raw = tel[SIGNAL_COLUMNS].isna().mean()
    tel, invalid_report = remove_invalid_values(tel)
    tel, spikes_removed = remove_spikes(tel)
    tel = fill_missing(tel)
    missing_after = tel[SIGNAL_COLUMNS].isna().mean()
    tel = add_motion(tel)
    tel = split_trips(tel)
    trips = build_trip_table(tel)

    missing = pd.DataFrame({"missing_raw_pct": missing_raw, "missing_after_fill_pct": missing_after}).mul(100).round(2)
    quality = missing.rename_axis("column").reset_index().merge(invalid_report, on="column", how="left")
    params = {
        "raw_readings": int(raw_summary["readings"].sum()),
        "impossible_timestamps": int(raw_summary["impossible_timestamps"].sum()),
        "rows_10s": len(tel),
        "trips": len(trips),
        "spikes_removed": spikes_removed,
    }

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    tel.to_parquet(PROCESSED_DIR / "telemetry_10s.parquet", index=False)
    trips.to_parquet(PROCESSED_DIR / "trips.parquet", index=False)
    trips.to_csv(PROCESSED_DIR / "trips.csv", index=False)
    quality.to_csv(PROCESSED_DIR / "data_quality_report.csv", index=False)
    (PROCESSED_DIR / "pipeline_params.json").write_text(json.dumps(params, indent=2))   # written last

tel = pd.read_parquet(PROCESSED_DIR / "telemetry_10s.parquet")
trips = pd.read_parquet(PROCESSED_DIR / "trips.parquet")
quality = pd.read_csv(PROCESSED_DIR / "data_quality_report.csv")
params = json.loads((PROCESSED_DIR / "pipeline_params.json").read_text())

print(f"10-second rows: {len(tel):,} ({(tel['state'] == 'driving').sum():,} while driving)")
print(f"Trips: {len(trips):,}")

10-second rows: 911,145 (290,380 while driving)
Trips: 2,389


## 1.12 Checking the cleaned data
The next cell raises an error if the data has duplicate rows, dates outside 2020-2024, trips without movement or shared between cars, or values outside their valid ranges.

In [15]:
trip_rows = tel[tel["trip_id"].notna()]
checks = {
    "one row per vehicle per 10 s": not tel.duplicated(["vehicle_id", "ts"]).any(),
    "all timestamps between 2020 and 2024": tel["ts"].between("2020-01-01", "2024-12-31").all(),
    "every trip has a moving row": (trip_rows.groupby("trip_id")["speed"].max() > 2).all(),
    "every trip belongs to one vehicle": (trip_rows.groupby("trip_id")["vehicle_id"].nunique() == 1).all(),
    "trip table matches the telemetry": set(trips["trip_id"]) == set(trip_rows["trip_id"]),
    "all values inside the valid ranges": all(tel[c].dropna().between(*r).all() for c, r in VALID_RANGES.items()),
}
assert all(checks.values()), checks
pd.Series(checks, name="passed")

one row per vehicle per 10 s            True
all timestamps between 2020 and 2024    True
every trip has a moving row             True
every trip belongs to one vehicle       True
trip table matches the telemetry        True
all values inside the valid ranges      True
Name: passed, dtype: bool

In [16]:
duckdb.sql("""
    SELECT vehicle_id,
           COUNT(*) AS trips,
           ROUND(SUM(distance_km)) AS km,
           MIN(start_ts)::DATE AS first_trip,
           MAX(start_ts)::DATE AS last_trip
    FROM trips
    GROUP BY vehicle_id
    ORDER BY trips DESC
""").df()

,vehicle_id,trips,km,first_trip,last_trip
0,ID2,836,18089.0,2021-07-26,2023-10-10
1,ID1,511,15448.0,2021-02-02,2023-06-07
2,CUP1,395,4049.0,2022-11-15,2023-04-24
3,CUP4,226,2333.0,2022-11-23,2023-04-06
4,CUP5,188,1343.0,2022-11-10,2023-04-23
5,CUP2,187,1859.0,2022-11-21,2023-04-21
6,CUP3,46,541.0,2022-11-21,2023-04-06


Trip counts differ a lot between cars, and the CUPRAs were only logged in winter and spring. Later models therefore split and shuffle data by car.

In [17]:
print(f"Raw readings: {params['raw_readings']:,}")
print(f"Readings with impossible timestamps removed: {params['impossible_timestamps']:,}")
print("One-reading temperature spikes removed:", params["spikes_removed"])

quality.rename(columns={"column": "signal", "values": "values before cleaning",
                        "missing_raw_pct": "% missing (raw)", "missing_after_fill_pct": "% missing (after filling)"}) \
    .sort_values("% missing (after filling)").reset_index(drop=True)

Raw readings: 98,061,214
Readings with impossible timestamps removed: 812
One-reading temperature spikes removed: {'bat_inlet_temp': 18526, 'bat_outlet_temp': 19599, 'coolant_temp': 104, 'motor_temp': 86, 'rotor_temp': 169, 'ambient_temp': 42}


,signal,% missing (raw),% missing (after filling),values before cleaning,placeholder_zeros,out_of_range
0,hv_voltage,2.20,1.25,891059.0,2.0,10.0
1,soc,5.12,3.98,864494.0,30.0,1.0
2,ambient_temp,32.30,27.52,616868.0,0.0,0.0
3,bat_outlet_temp,32.84,27.86,611892.0,21680.0,7108.0
4,bat_inlet_temp,32.81,27.92,612241.0,20948.0,7170.0
5,coolant_temp,28.88,28.21,648007.0,0.0,0.0
6,rotor_temp,28.90,28.22,647855.0,0.0,0.0
7,motor_temp,30.32,29.64,634906.0,0.0,0.0
8,hv_aux_power,31.32,31.10,625785.0,0.0,4.0
9,speed,51.42,51.44,442612.0,0.0,132.0


- "% missing" includes parked and charging time, so most sensors are empty part of the time
- `interior_temp` and the two PTC heater signals are almost always empty

## 1.13 Trips used for the energy analysis
- **`energy`**: trips with a reliable energy value (`energy_valid_5km`) and an outside temperature reading
- **`driving`**: all 10-second rows recorded during trips

In [18]:
energy = trips[trips["energy_valid_5km"] & trips["ambient_temp_mean"].notna()].copy()
driving = tel[tel["state"] == "driving"]

display(pd.Series({
    "All trips": len(trips),
    "At least 5 km": (trips["distance_km"] >= 5).sum(),
    "... and believable energy use": trips["energy_valid_5km"].sum(),
    "... and an outside temperature reading": len(energy),
}, name="trips"))

print("Raw energy range:", round(trips["soc_per_100km"].min(), 1), "to", round(trips["soc_per_100km"].max(), 1),
      "SOC % per 100 km")
print("Driving rows with a speed reading:", round(driving["speed"].notna().mean() * 100, 1), "%")
print("Driving rows missing pack temperature (after interpolation):", round(driving["bat_temp"].isna().mean() * 100), "%")

All trips                                 2389
At least 5 km                             1388
... and believable energy use             1374
... and an outside temperature reading    1326
Name: trips, dtype: int64

Raw energy range: -111.6 to 331.1 SOC % per 100 km
Driving rows with a speed reading: 99.9 %
Driving rows missing pack temperature (after interpolation): 57 %


Energy values below 0 or above 100 % per 100 km come from rounding on short trips, which is why only trips of at least 5 km are used.

# Part 2: Exploratory Data Analysis

## 2.1 Energy use vs outside temperature (RQ1, RQ3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(energy["ambient_temp_mean"], energy["soc_per_100km"], s=8, alpha=0.4)
plt.title("Energy use vs outside temperature")
plt.xlabel("Outside temperature (°C)")
plt.ylabel("SOC % per 100 km")
plt.show()

print("Spearman (temperature):", round(spearmanr(energy["ambient_temp_mean"], energy["soc_per_100km"]).statistic, 2))
print("Spearman (mean speed):", round(spearmanr(energy["speed_mean_moving"], energy["soc_per_100km"]).statistic, 2))

- Colder trips use more energy; average speed barely matters
- Outside temperature is therefore used as a feature in RQ3 and as a control in the hypothesis tests (H1, H2)

## 2.2 Energy use by month and car model (RQ1, RQ3)

In [ ]:
energy["model"] = np.where(energy["vehicle_id"].str.startswith("ID"), "VW ID.3", "CUPRA Born")

plt.figure(figsize=(11, 5))
sns.boxplot(data=energy, x="start_month", y="soc_per_100km", hue="model", showfliers=False)
plt.title("Energy use by month")
plt.xlabel("Month")
plt.ylabel("SOC % per 100 km")
plt.xticks(range(12), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
plt.legend(title="Car model", loc="upper center")
plt.show()

- Energy use is highest in winter
- The CUPRAs were only logged November to May, so car and season are mixed up
- The energy model therefore includes month and car model, and the hypothesis tests shuffle data within each car

## 2.3 Correlation between trip features (RQ1)

In [ ]:
candidates = ["speed_mean_moving", "speed_std", "speed_max", "stop_fraction",
              "accel_abs_mean", "accel_p95", "decel_p05", "duration_min", "distance_km"]

plt.figure(figsize=(8, 6))
sns.heatmap(trips[candidates].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation between trip features")
plt.show()

- Distance and duration are strongly correlated (0.93), as are max speed and speed variation (0.93), so clustering keeps only duration and speed variation
- The three acceleration features are also correlated (~0.8) but measure different things (overall, hard acceleration, hard braking), so all three are kept

## 2.4 Battery temperature coverage and 5-minute change (RQ2, RQ4)

In [ ]:
observed_share = driving.groupby("vehicle_id")["bat_temp_observed"].mean().mul(100).sort_values()

real = tel.loc[tel["trip_id"].notna() & tel["bat_temp_observed"], ["trip_id", "ts", "bat_temp"]]
later = real.rename(columns={"ts": "ts_later", "bat_temp": "bat_temp_later"}).sort_values("ts_later")
pairs = pd.merge_asof(real.assign(ts_later=real["ts"] + pd.Timedelta(minutes=5)).sort_values("ts_later"), later,
                      on="ts_later", by="trip_id", direction="nearest", tolerance=pd.Timedelta(seconds=20))
change_5min = (pairs["bat_temp_later"] - pairs["bat_temp"]).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
observed_share.plot.barh(ax=axes[0], color="tab:blue")
axes[0].set_title("Driving rows with a real pack-temperature reading")
axes[0].set_xlabel("% of driving rows")
axes[0].set_ylabel("")
axes[1].hist(change_5min.clip(-3, 4), bins=70, color="tab:orange")
axes[1].set_title("Change in pack temperature over 5 minutes (real readings)")
axes[1].set_xlabel("°C")
axes[1].set_ylabel("Number of pairs")
plt.tight_layout()
plt.show()

print("Real readings in driving rows:", round(driving["bat_temp_observed"].mean() * 100, 1), "%")
print("5-minute pairs:", len(change_5min))
print("Share of 5-minute changes within ±0.5 °C:", round((change_5min.abs() <= 0.5).mean() * 100, 1), "%")
print("95th percentile of the 5-minute rise:", round(change_5min.quantile(0.95), 2), "°C")

- Only 6.1% of driving rows have a real battery temperature reading; the rest is interpolated
- Battery temperature changes slowly: 81% of 5-minute changes are within ±0.5 °C
- So RQ2 uses only real readings, and "temperature stays the same" is the baseline to beat

# Part 3: Modeling and Hypothesis Testing
One section per research question, so each can be worked on separately. Every section needs Part 1 and 3.0 to have run; apart from that, sections do not depend on each other (RQ4 will reuse the features from 3.2.1).

## 3.0 Shared helpers
Run this first. It defines the cross-validation and permutation-test functions used by every section.

**Hypothesis tests** use permutation tests with 10,000 shuffles. Each trip is one data point, and labels are shuffled within each vehicle.

In [ ]:
def regression_scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "R2": r2_score(y_true, y_pred)}


def cross_validate(models, X, y, groups, splitter):
    """
    Out-of-fold predictions for every model, using the same folds for all of them.
    """
    predictions = {name: np.zeros(len(y)) for name in models}
    for train, test in splitter.split(X, y, groups):
        for name, make_model in models.items():
            model = make_model().fit(X.iloc[train], y.iloc[train])
            predictions[name][test] = model.predict(X.iloc[test])
    return predictions

In [ ]:
rng = np.random.default_rng(0)


def permutation_test(df, label_col, value_col, statistic, n_perm=10_000):
    labels = df[label_col].to_numpy()
    values = df[value_col].to_numpy()
    vehicles = df["vehicle_id"].to_numpy()
    observed = statistic(labels, values)

    null_stats = []
    for _ in range(n_perm):
        shuffled = labels.copy()
        for v in np.unique(vehicles):
            idx = np.where(vehicles == v)[0]
            shuffled[idx] = rng.permutation(labels[idx])
        null_stats.append(statistic(shuffled, values))

    p_value = (np.sum(np.array(null_stats) >= observed) + 1) / (n_perm + 1)
    return observed, p_value


def diff_in_means(labels, values):
    return values[labels == 1].mean() - values[labels == 0].mean()


def sd_of_group_means(labels, values):
    return np.std([values[labels == g].mean() for g in np.unique(labels)])

In [ ]:
def paired_permutation_test(errors_baseline, errors_model, groups, n_perm=10_000):
    """
    One-sided test that the model has a lower absolute error than the baseline.
    Errors are averaged per group (trip), then the sign of each group's difference is flipped at random.
    """
    diff = pd.Series(np.abs(errors_baseline) - np.abs(errors_model)).groupby(np.asarray(groups)).mean().to_numpy()
    observed = diff.mean()
    null = (rng.choice([-1, 1], size=(n_perm, len(diff))) * diff).mean(axis=1)
    return observed, (np.sum(null >= observed) + 1) / (n_perm + 1)

## 3.1 RQ1: Driving regimes
**Question:** can trips be grouped into driving regimes, and does energy use differ between them?

### 3.1.1 Clustering features

In [ ]:
cluster_features = ["speed_mean_moving", "speed_std", "stop_fraction",
                    "accel_abs_mean", "accel_p95", "decel_p05", "duration_min"]

X = trips[cluster_features].copy()
X["duration_min"] = np.log1p(X["duration_min"])
X_scaled = StandardScaler().fit_transform(X)

Duration is log-transformed because it is heavily skewed. All features are scaled because K-Means uses distances.

### 3.1.2 Baseline: speed bins

In [ ]:
def cluster_scores(labels):
    return {
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
    }


baseline_labels = pd.cut(trips["speed_mean_moving"], bins=[0, 30, 60, np.inf], labels=False)
cluster_scores(baseline_labels)

### 3.1.3 K-Means

In [ ]:
k_results = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_scaled)
    k_results.append({"k": k, **cluster_scores(labels)})

pd.DataFrame(k_results).set_index("k").round(3)

We use k = 4. It has the lowest Davies-Bouldin score, and k = 2 only separates short and long trips.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=50, random_state=0).fit(X_scaled)

regime_names = ["Short urban", "Dynamic urban", "Mixed suburban", "Highway"]
cluster_speed = trips.groupby(kmeans.labels_)["speed_mean_moving"].median().sort_values()
trips["regime"] = pd.Categorical(pd.Series(kmeans.labels_).map(dict(zip(cluster_speed.index, regime_names))),
                                 categories=regime_names)

pd.DataFrame({"Baseline": cluster_scores(baseline_labels), "K-Means": cluster_scores(kmeans.labels_)}).T.round(3)

In [ ]:
trips.groupby("regime", observed=True)[cluster_features].median().round(2)

In [ ]:
pcs = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=trips["regime"], s=10, alpha=0.6)
plt.title("Driving regimes (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

K-Means scores better than the speed-bin baseline on all three metrics. The regimes overlap a bit, so driving style is more of a spectrum.

### 3.1.4 Gaussian Mixture Model

Model 2 for RQ1. A Gaussian Mixture Model gives soft (probabilistic) assignments and, with full covariance, allows elongated clusters and correlated features, which fits the overlap seen in the PCA plot. We choose k with BIC and compare it with K-Means at k = 4.

In [ ]:
gmm_results = []
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=5, random_state=0).fit(X_scaled)
    gmm_results.append({"k": k, "BIC": gmm.bic(X_scaled), **cluster_scores(gmm.predict(X_scaled))})

pd.DataFrame(gmm_results).set_index("k").round(3)

In [ ]:
gmm4 = GaussianMixture(n_components=4, covariance_type="full", n_init=10, random_state=0).fit(X_scaled)
gmm_labels = gmm4.predict(X_scaled)
gmm_confidence = gmm4.predict_proba(X_scaled).max(axis=1)

kmeans_stability = [adjusted_rand_score(kmeans.labels_, KMeans(n_clusters=4, n_init=10, random_state=s).fit_predict(X_scaled))
                    for s in range(1, 11)]

print("K-Means stability across 10 random seeds (mean ARI):", round(np.mean(kmeans_stability), 3))
print("Agreement between K-Means and GMM (ARI):", round(adjusted_rand_score(kmeans.labels_, gmm_labels), 3))
print("GMM trips assigned with > 90% probability:", round((gmm_confidence > 0.9).mean() * 100, 1), "%")

cluster_comparison = pd.DataFrame({
    "Baseline: speed bins": cluster_scores(baseline_labels),
    "K-Means (k=4)": cluster_scores(kmeans.labels_),
    "GMM (k=4)": cluster_scores(gmm_labels),
}).T.round(3)
cluster_comparison

In [ ]:
pd.crosstab(trips["regime"], pd.Series(gmm_labels, name="GMM cluster", index=trips.index))

BIC is lowest at k = 7, so on likelihood alone the GMM would use more clusters, but those clusters are poorly separated (silhouette 0.13). At k = 4 the GMM scores worse than K-Means on all three metrics (silhouette 0.18 vs 0.27, Davies-Bouldin 2.32 vs 1.18). These metrics favour round clusters, so they lean towards K-Means; the crosstab is more telling. The GMM merges *Short urban* and *Dynamic urban* into one cluster and keeps *Mixed suburban* and *Highway* mostly separate (agreement ARI = 0.39). It assigns 77% of trips with more than 90% probability. K-Means is very stable (ARI = 0.998 across 10 random seeds).

**Decision:** we keep the K-Means regimes for the hypothesis tests because they are stable and easy to interpret, and report the GMM as a check: the urban vs suburban vs highway split is robust, while the line between the two urban regimes is soft.

### 3.1.5 H1: Cold trips use more energy
Outside temperature is the strongest driver of energy use (2.1), so we test it first; H2 then adjusts for it.
H0: cold (< 10 °C) and warm trips use the same energy.

In [ ]:
energy["cold"] = (energy["ambient_temp_mean"] < 10).astype(int)
diff, p_value = permutation_test(energy, "cold", "soc_per_100km", diff_in_means)

print("Cold - warm:", round(diff, 2), "SOC % per 100 km")
print("p-value:", round(p_value, 4))

### 3.1.6 H2: Energy use differs by driving regime
H0: energy use does not depend on regime. We also test energy adjusted for temperature.

In [ ]:
energy = energy.drop(columns=["regime"], errors="ignore").merge(trips[["trip_id", "regime"]], on="trip_id")
energy["regime_code"] = energy["regime"].cat.codes

slope, intercept = np.polyfit(energy["ambient_temp_mean"], energy["soc_per_100km"], 1)
energy["soc_adjusted"] = energy["soc_per_100km"] - (intercept + slope * energy["ambient_temp_mean"])

_, p_raw = permutation_test(energy, "regime_code", "soc_per_100km", sd_of_group_means)
_, p_adjusted = permutation_test(energy, "regime_code", "soc_adjusted", sd_of_group_means)

print("p-value (raw):", round(p_raw, 4))
print("p-value (temperature-adjusted):", round(p_adjusted, 4))
energy.groupby("regime", observed=True)[["soc_per_100km", "soc_adjusted", "ambient_temp_mean"]].mean().round(2)

Cold trips use clearly more energy. Regime only matters once temperature is accounted for, because highway trips happen in warmer weather.

**Reading H2 carefully:** the statistic (spread of the regime means) says that at least one regime differs, not which one. After adjusting for temperature, highway trips use about 2.3 SOC %/100 km more than expected and mixed suburban trips the least, which fits air drag at high speed. The temperature adjustment is linear, which is a simplification, since heating mostly matters in the cold.

### 3.1.7 Robustness of H1
The 10 °C cut-off is a choice, so we repeat H1 at 5 °C and 15 °C, and also test the continuous relationship (Spearman correlation between outside temperature and energy, with temperatures shuffled within each vehicle).

In [ ]:
robustness = []
for cutoff in [5, 10, 15]:
    labels = energy.assign(cold=(energy["ambient_temp_mean"] < cutoff).astype(int))
    diff, p = permutation_test(labels, "cold", "soc_per_100km", diff_in_means)
    robustness.append({"test": f"cold < {cutoff} °C vs warm", "statistic": round(diff, 2), "p-value": p})

negative_spearman = lambda temps, values: -spearmanr(temps, values).statistic
rho, p = permutation_test(energy, "ambient_temp_mean", "soc_per_100km", negative_spearman)
robustness.append({"test": "Spearman correlation (temperature vs energy)", "statistic": round(-rho, 3), "p-value": p})
pd.DataFrame(robustness)

The result does not depend on the 10 °C cut-off: the cold–warm gap is 8.3, 7.8 and 7.0 SOC %/100 km at 5, 10 and 15 °C, and the Spearman correlation is −0.56. Every p-value is 0.0001, the smallest possible with 10,000 shuffles.

## 3.2 RQ2: Battery temperature 5 minutes ahead
**Question:** can recent operating conditions predict battery temperature 5 minutes ahead, beyond the current temperature?

### 3.2.1 Features

In [ ]:
HORIZON = pd.Timedelta(minutes=5)

d = tel[tel["trip_id"].notna()].sort_values(["trip_id", "ts"]).reset_index(drop=True)
d["accel_abs"] = d["accel_ms2"].abs()
d["is_id3"] = d["vehicle_id"].str.startswith("ID").astype(int)
d["trip_elapsed_min"] = d["trip_elapsed_s"] / 60

# Current temperature = last REAL reading at or before t (no interpolation, so no future information)
d["temp_obs"] = d["bat_temp"].where(d["bat_temp_observed"])
d["obs_ts"] = d["ts"].where(d["bat_temp_observed"])
d["temp_now"] = d.groupby("trip_id")["temp_obs"].ffill()
d["temp_age_s"] = (d["ts"] - d.groupby("trip_id")["obs_ts"].ffill()).dt.total_seconds()

# Operating conditions over the past 5 minutes (rolling windows only look backwards)
past = d.set_index("ts").groupby("trip_id")
d["speed_mean_5m"] = past["speed"].rolling("300s").mean().to_numpy()
d["speed_max_5m"] = past["speed"].rolling("300s").max().to_numpy()
d["accel_abs_5m"] = past["accel_abs"].rolling("300s").mean().to_numpy()
d["aux_power_5m"] = past["hv_aux_power"].rolling("300s").mean().to_numpy()


def value_before(df, col, delta):
    """
    Value of `col` in the same trip `delta` earlier (within 60 s), or NaN.
    """
    left = df[["trip_id"]].assign(ts_past=df["ts"] - delta).reset_index().sort_values("ts_past")
    right = df[["trip_id", "ts", col]].dropna().rename(columns={"ts": "ts_past", col: "past"}).sort_values("ts_past")
    merged = pd.merge_asof(left, right, on="ts_past", by="trip_id", direction="backward", tolerance=pd.Timedelta(seconds=60))
    return merged.set_index("index")["past"].reindex(df.index)


d["temp_change_5m"] = d["temp_now"] - value_before(d, "temp_now", HORIZON)
d["outlet_change_5m"] = d["bat_outlet_temp"] - value_before(d, "bat_outlet_temp", HORIZON)

# Target: the real pack-temperature reading 5 minutes later (within ±20 s), same trip
real = d.loc[d["bat_temp_observed"], ["trip_id", "ts", "bat_temp"]]
real = real.rename(columns={"ts": "ts_future", "bat_temp": "temp_future"}).sort_values("ts_future")
left = d[["trip_id"]].assign(ts_future=d["ts"] + HORIZON).reset_index().sort_values("ts_future")
target = pd.merge_asof(left, real, on="ts_future", by="trip_id", direction="nearest", tolerance=pd.Timedelta(seconds=20))
d["temp_future"] = target.set_index("index")["temp_future"].reindex(d.index)

TEMP_FEATURES = ["temp_now", "temp_age_s", "temp_change_5m", "speed_mean_5m", "speed_max_5m", "accel_abs_5m",
                 "aux_power_5m", "ambient_temp", "bat_outlet_temp", "outlet_change_5m", "soc", "trip_elapsed_min", "is_id3"]

temp_data = d[d["temp_future"].notna() & d["temp_now"].notna() & (d["temp_age_s"] <= 600) & (d["trip_elapsed_s"] >= 300)]
temp_data = temp_data[temp_data["ts"].dt.second % 30 == 0].reset_index(drop=True)   # one row every 30 s

print("Forecasting rows:", len(temp_data), "| trips:", temp_data["trip_id"].nunique(),
      "| vehicles:", temp_data["vehicle_id"].nunique())
temp_data[TEMP_FEATURES + ["temp_future"]].describe().T.round(2)

Each row is a moment *t* during a trip. Every feature uses only data up to *t*: the last real pack-temperature reading and its age, the temperature change over the past 5 minutes, speed, acceleration and heater/AC power over the past 5 minutes, outside and coolant temperature, SOC, time into the trip, and car model. The target is the real reading 5 minutes later. We start at 5 minutes into the trip so every row has a full 5-minute history, and keep one row every 30 seconds to limit near-duplicate rows.

### 3.2.2 Models

**Approach.** The models predict the *change* in temperature over the next 5 minutes; the forecast is current temperature + predicted change.

| | Model | Why |
| --- | --- | --- |
| Baseline | Persistence: temperature in 5 min = current temperature | The standard forecasting baseline; hard to beat because temperature changes slowly |
| Reference | Linear regression (Ridge) | Interpretable; shows how much a linear effect of the features explains |
| Model 1 | Random Forest | Captures non-linear effects and interactions (e.g. speed × outside temperature); robust with little tuning |
| Model 2 | XGBoost | Usually the most accurate on tabular data; handles missing values natively |

**Validation:** 5-fold GroupKFold by trip, so rows from the same trip are never in both training and test data (neighbouring rows are almost identical). **Metrics:** MAE and RMSE in °C, R² of the temperature, and R² of the 5-minute change (the harder number, since predicting temperature ≈ current temperature already gives a high R²). Every model is scored on the same folds as the baseline.

In [ ]:
X_temp = temp_data[TEMP_FEATURES]
y_change = temp_data["temp_future"] - temp_data["temp_now"]

temp_models = {
    "Baseline: persistence": lambda: DummyRegressor(strategy="constant", constant=0.0),
    "Linear regression (Ridge)": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                           RandomForestRegressor(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)),
    "XGBoost": lambda: XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8,
                                    colsample_bytree=0.8, n_jobs=-1, random_state=0),
}

temp_pred = cross_validate(temp_models, X_temp, y_change, temp_data["trip_id"], GroupKFold(n_splits=5))

temp_results = pd.DataFrame({name: regression_scores(temp_data["temp_future"], temp_data["temp_now"] + p)
                             for name, p in temp_pred.items()}).T
temp_results["R2 of 5-min change"] = [r2_score(y_change, p) for p in temp_pred.values()]
temp_results["MAE vs baseline"] = temp_results["MAE"] / temp_results.loc["Baseline: persistence", "MAE"] - 1
temp_results.round(3)

Both models beat persistence on the same folds. XGBoost has the lowest error: **MAE 0.31 °C vs 0.55 °C for persistence (43% lower)** and RMSE 0.52 vs 0.96 °C. It explains 69% of the variance in the 5-minute change, while persistence explains none. Random Forest is close (MAE 0.35 °C). The linear model improves by only 18%, so the effects are non-linear. R² of the temperature itself is above 0.98 for every model, including persistence, which is why we judge models on MAE/RMSE and on R² of the change.

### 3.2.3 Unseen vehicles

In [ ]:
# Harder check: train on 6 cars, test on the 7th (leave one vehicle out)
lovo_temp = cross_validate({k: temp_models[k] for k in ["Baseline: persistence", "XGBoost"]},
                           X_temp, y_change, temp_data["vehicle_id"], LeaveOneGroupOut())
pd.DataFrame({name: regression_scores(temp_data["temp_future"], temp_data["temp_now"] + p)
              for name, p in lovo_temp.items()}).T.round(3)

On a car that was not in the training data, XGBoost still reaches MAE 0.35 °C vs 0.55 °C for persistence, so the model generalises across vehicles rather than learning one car's behaviour.

### 3.2.4 Feature importance

In [ ]:
train, test = next(GroupKFold(n_splits=5).split(X_temp, y_change, temp_data["trip_id"]))
xgb_temp = temp_models["XGBoost"]().fit(X_temp.iloc[train], y_change.iloc[train])
importance = permutation_importance(xgb_temp, X_temp.iloc[test], y_change.iloc[test], n_repeats=5,
                                    scoring="neg_mean_absolute_error", random_state=0)
temp_importance = pd.Series(importance.importances_mean, index=TEMP_FEATURES).sort_values()

temp_importance.plot.barh(figsize=(7, 5), color="tab:blue")
plt.title("RQ2: which features matter (XGBoost, permutation importance)")
plt.xlabel("Increase in MAE when the feature is shuffled (°C)")
plt.show()

The current temperature matters most, as expected. Next come recent maximum and mean speed, coolant outlet temperature and outside temperature: higher speeds heat the pack, while the coolant and outside air set how fast it cools.

**Trade-offs:** XGBoost is the most accurate but the least transparent; permutation importance and the Ridge model give the interpretable view. With 8,484 rows from 912 trips, tree models are a good fit; an LSTM would need more data and stays a stretch goal.

### 3.2.5 H3: Recent operating conditions improve the 5-minute temperature forecast (RQ2)
H0: the best model is no more accurate than persistence (which uses only the current temperature). We compare the absolute errors of the two on the same out-of-fold rows, average the difference per trip, and run a sign-flip permutation test over trips (10,000 random sign flips). Trips, not rows, are the unit, because rows within a trip are not independent.

In [ ]:
best_temp_model = temp_results.drop(index="Baseline: persistence")["MAE"].idxmin()
gain, p_h3 = paired_permutation_test(y_change - temp_pred["Baseline: persistence"],
                                     y_change - temp_pred[best_temp_model], temp_data["trip_id"])
print("Best model:", best_temp_model)
print("Mean MAE reduction per trip:", round(gain, 3), "°C")
print("p-value:", round(p_h3, 4))

XGBoost lowers the absolute error by 0.22 °C per trip on average, p = 0.0001, so we reject H0: recent driving conditions add information beyond the current temperature.

## 3.3 RQ3: Rest-of-trip energy use
**Question:** can the first 5 minutes of a trip predict energy use for the rest of it?

### 3.3.1 Features

In [ ]:
EARLY_S = 300   # the first 5 minutes of a trip

early = tel[tel["trip_id"].notna() & (tel["trip_elapsed_s"] < EARLY_S)].copy()
early["moving"] = early["speed"] > 2
g = early.groupby("trip_id")
early_features = pd.DataFrame({
    "early_speed_mean": g["speed"].mean(),
    "early_speed_std": g["speed"].std(),
    "early_speed_max": g["speed"].max(),
    "early_stop_fraction": 1 - g["moving"].mean(),
    "early_accel_abs": g["accel_ms2"].apply(lambda s: s.abs().mean()),
    "early_accel_p95": g["accel_ms2"].quantile(0.95),
    "early_dist_km": g["dist_km"].sum(),
    "early_ambient_temp": g["ambient_temp"].mean(),
    "early_aux_power": g["hv_aux_power"].mean(),
    "early_outlet_temp": g["bat_outlet_temp"].mean(),
    "soc_start": g["soc"].first(),
    "soc_at_5min": g["soc"].last(),
}).reset_index()

rest = trips[["trip_id", "vehicle_id", "start_ts", "distance_km", "duration_min", "soc_end"]].merge(early_features, on="trip_id")
rest["early_soc_used"] = rest["soc_start"] - rest["soc_at_5min"]
rest["rest_km"] = rest["distance_km"] - rest["early_dist_km"]
rest["rest_soc_per_100km"] = 100 * (rest["soc_at_5min"] - rest["soc_end"]) / rest["rest_km"]
rest["start_hour"] = rest["start_ts"].dt.hour
rest["month_sin"] = np.sin(2 * np.pi * rest["start_ts"].dt.month / 12)
rest["month_cos"] = np.cos(2 * np.pi * rest["start_ts"].dt.month / 12)
rest["is_id3"] = rest["vehicle_id"].str.startswith("ID").astype(int)

ENERGY_FEATURES = ["early_speed_mean", "early_speed_std", "early_speed_max", "early_stop_fraction", "early_accel_abs",
                   "early_accel_p95", "early_dist_km", "early_ambient_temp", "early_aux_power", "early_outlet_temp",
                   "soc_start", "early_soc_used", "start_hour", "month_sin", "month_cos", "is_id3"]

energy_data = rest[(rest["duration_min"] > 10) & (rest["rest_km"] >= 5)
                   & rest["rest_soc_per_100km"].between(5, 100)].reset_index(drop=True)
print("Trips for the rest-of-trip model:", len(energy_data), "| vehicles:", energy_data["vehicle_id"].nunique())
energy_data[ENERGY_FEATURES + ["rest_soc_per_100km"]].describe().T.round(2)

One row per trip. Features describe only the first 5 minutes (speed, stops, acceleration, distance, outside temperature, heater/AC power, coolant temperature, SOC used so far) plus things known at the start (time of day, season as sine/cosine of the month, car model). The target is SOC % per 100 km for the **rest** of the trip. We keep trips longer than 10 minutes with at least 5 km left, so SOC rounding does not dominate. Whole-trip columns (including the driving regime, which is clustered on the full trip) are not used, because they would leak the future.

### 3.3.2 Models

**Approach.**

| | Model | Why |
| --- | --- | --- |
| Baseline | Training-set mean | What we would predict knowing nothing about the trip |
| Baseline from the proposal | Lasso (L1-regularised linear regression, alpha chosen by inner 5-fold CV) | Interpretable; shows which early-trip signals matter linearly and drops weak ones |
| Model 1 | Random Forest | Non-linear effects, e.g. heating only matters below about 10 °C |
| Model 2 | XGBoost | Strong on small tabular data with regularisation |

**Validation:** 5-fold cross-validation with shuffling (each trip is one row, so trips never overlap between folds), plus a leave-one-vehicle-out check. **Metrics:** MAE and RMSE in SOC %/100 km, and R². All models use the same folds.

In [ ]:
X_energy = energy_data[ENERGY_FEATURES]
y_energy = energy_data["rest_soc_per_100km"]

energy_models = {
    "Baseline: training mean": lambda: DummyRegressor(strategy="mean"),
    "Lasso (proposal baseline)": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                       LassoCV(cv=5, random_state=0)),
    "Random Forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                           RandomForestRegressor(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0)),
    "XGBoost": lambda: XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8,
                                    colsample_bytree=0.8, min_child_weight=5, n_jobs=-1, random_state=0),
}

energy_pred = cross_validate(energy_models, X_energy, y_energy, None,   # one row per trip, so plain shuffled K-fold
                             KFold(n_splits=5, shuffle=True, random_state=0))

energy_results = pd.DataFrame({name: regression_scores(y_energy, p) for name, p in energy_pred.items()}).T
energy_results["MAE vs baseline"] = energy_results["MAE"] / energy_results.loc["Baseline: training mean", "MAE"] - 1
energy_results.round(3)

Predicting the rest of a trip from its first 5 minutes is harder. XGBoost has the lowest error: **MAE 4.75 vs 5.97 SOC %/100 km for the mean baseline (20% lower)**, R² = 0.32. Random Forest (4.79) and Lasso (5.02) are close, so a simple linear model already captures most of the gain. Part of the remaining error is noise in the target: SOC is logged in 0.4% steps, so a 10 km remainder carries up to ±4 SOC %/100 km of rounding alone.

### 3.3.3 Unseen vehicles

In [ ]:
lovo_energy = cross_validate({k: energy_models[k] for k in ["Baseline: training mean", "Lasso (proposal baseline)", "XGBoost"]},
                             X_energy, y_energy, energy_data["vehicle_id"], LeaveOneGroupOut())
pd.DataFrame({name: regression_scores(y_energy, p) for name, p in lovo_energy.items()}).T.round(3)

On an unseen car every model loses accuracy (XGBoost MAE 5.25, R² = 0.17), but still beats the baseline (6.06). The cars differ (two models, different seasons logged), so we report both validation schemes.

### 3.3.4 Feature importance

In [ ]:
lasso = energy_models["Lasso (proposal baseline)"]().fit(X_energy, y_energy)
lasso_coef = pd.Series(lasso[-1].coef_, index=ENERGY_FEATURES)

train, test = next(KFold(n_splits=5, shuffle=True, random_state=0).split(X_energy))
xgb_energy = energy_models["XGBoost"]().fit(X_energy.iloc[train], y_energy.iloc[train])
importance = permutation_importance(xgb_energy, X_energy.iloc[test], y_energy.iloc[test], n_repeats=20,
                                    scoring="neg_mean_absolute_error", random_state=0)
energy_importance = pd.Series(importance.importances_mean, index=ENERGY_FEATURES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
lasso_coef.sort_values().plot.barh(ax=axes[0], color="tab:green")
axes[0].set_title("Lasso coefficients (standardised features)")
axes[0].set_xlabel("SOC %/100 km per 1 SD")
energy_importance.sort_values().plot.barh(ax=axes[1], color="tab:blue")
axes[1].set_title("XGBoost permutation importance")
axes[1].set_xlabel("Increase in MAE when shuffled (SOC %/100 km)")
plt.tight_layout()
plt.show()
print("Features dropped by Lasso:", list(lasso_coef[lasso_coef == 0].index))

Both views agree that outside temperature is the main driver, with heater/AC power next. XGBoost also uses coolant temperature, and Lasso also uses early speed and the SOC already used. Lasso keeps 7 of the 16 features. Its error is only about 6% higher than XGBoost's.

**Trade-off:** we use Lasso to explain the effects and XGBoost for the best accuracy.

### 3.3.5 H4: Early-trip behaviour predicts rest-of-trip energy (RQ3)
H0: the best model is no more accurate than predicting the training mean. Same sign-flip test, with each trip as one unit.

In [ ]:
best_energy_model = energy_results.drop(index="Baseline: training mean")["MAE"].idxmin()
gain, p_h4 = paired_permutation_test(y_energy - energy_pred["Baseline: training mean"],
                                     y_energy - energy_pred[best_energy_model], energy_data["trip_id"])
print("Best model:", best_energy_model)
print("Mean MAE reduction per trip:", round(gain, 2), "SOC %/100 km")
print("p-value:", round(p_h4, 4))

XGBoost lowers the absolute error by 1.21 SOC %/100 km per trip on average, p = 0.0001, so we reject H0: the first 5 minutes carry real information about the rest of the trip, although most of the variance (R² = 0.32) is still unexplained.

## 3.4 RQ4: Thermal-demand classification
**Question:** given the current operating conditions, will the battery enter a thermally demanding state within the next 5 minutes?

**Plan (from the proposal):**
- **Target:** a binary label $D_T$ = 1 for a high temperature or a large temperature rise within the next 5 minutes.
- **Inputs:** battery temperature, speed, SOC, battery voltage and outside temperature (the data has no motor RPM; speed and acceleration replace it).
- **Models:** Logistic Regression as the baseline, compared against Random Forest and XGBoost, validated with GroupKFold by trip.
- **Metrics:** precision, recall, F1 and ROC-AUC.
- **Test:** permutation test of whether recent history predicts $D_T$.

**Starting point:** 3.2.1 already builds leakage-safe features (`temp_data`, `TEMP_FEATURES`) and the real pack-temperature reading 5 minutes later (`temp_future`), so the label can be built from them, e.g. `temp_future - temp_now >= 1.1` (large rise) or `temp_future >= 30.3` (high temperature). Use only real readings (`bat_temp_observed`), never interpolated ones. Do not use the trip's driving regime as an input: it is computed from the whole trip and would leak the future.

## 3.5 RQ5 (extension): Usage patterns and battery health
**Question:** do long-term usage patterns relate to battery State of Health (SOH)?

**Data:** needs a second dataset, the [TUM Electric Vehicle Usage and State of Health data](https://data.gov.de/suche/daten/electric-vehicle-usage-and-state-of-health-dataf7e88) (1-minute samples including SOH). Part 1 does not download it.

**Plan (from the proposal):** summarize usage per vehicle (share of time in each driving regime, average speed, battery temperature exposure, SOC range used, mileage); Lasso Regression as the baseline vs Random Forest, evaluated with RMSE and MAE; feature importance; a permutation test that shuffles SOH labels between vehicles. With only 7 vehicles, results are exploratory.

# Part 4: Results Summary

## 4.1 Model comparison

In [ ]:
comparison = pd.concat({
    "RQ2: battery temperature in 5 min (°C)": temp_results[["MAE", "RMSE", "R2", "MAE vs baseline"]],
    "RQ3: rest-of-trip energy (SOC %/100 km)": energy_results[["MAE", "RMSE", "R2", "MAE vs baseline"]],
}).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (title, table) in zip(axes, [("RQ2: battery temperature, MAE (°C)", temp_results),
                                     ("RQ3: rest-of-trip energy, MAE (SOC %/100 km)", energy_results)]):
    colors = ["tab:gray" if name.startswith("Baseline") else "tab:blue" for name in table.index]
    table["MAE"].plot.barh(ax=ax, color=colors)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel("MAE (lower is better)")
plt.tight_layout()
plt.show()
comparison

| Task | Baseline | Best model | MAE reduction |
| --- | --- | --- | --- |
| RQ2: battery temperature in 5 min | Persistence, MAE 0.55 °C | XGBoost, MAE 0.31 °C | 43% |
| RQ3: rest-of-trip energy | Training mean, MAE 5.97 SOC %/100 km | XGBoost, MAE 4.75 SOC %/100 km | 20% |

**How we compare models:** the same rows and folds for every model; MAE as the main metric (in the target's units and robust to outliers), RMSE to check large errors, and R² for explained variance. Whether the improvement over the baseline is real is tested in 3.2.5 and 3.3.5. Hyperparameters are fixed, reasonable settings for now; tuning with nested cross-validation is the next step.

## 4.2 Hypothesis tests
We run four main tests (H1–H4), so we use a Bonferroni-corrected significance level of 0.05 / 4 = 0.0125.

In [ ]:
pd.DataFrame([
    {"hypothesis": "H1: cold trips (< 10 °C) use more energy", "p-value": p_value},
    {"hypothesis": "H2: energy differs by regime (temperature-adjusted)", "p-value": p_adjusted},
    {"hypothesis": "H3: recent conditions improve the 5-min temperature forecast", "p-value": p_h3},
    {"hypothesis": "H4: early-trip behaviour predicts rest-of-trip energy", "p-value": p_h4},
]).assign(**{"significant at 0.0125": lambda t: t["p-value"] < 0.0125}).round(4)

## 4.3 Results so far

| RQ | Result |
| --- | --- |
| RQ1: driving regimes | 4 stable K-Means regimes (silhouette 0.27 vs 0.13 for the speed-bin baseline). Cold trips (< 10 °C) use 7.8 SOC %/100 km more, robust to the cut-off (H1, p = 0.0001), and after adjusting for temperature, energy use differs by regime (H2, p = 0.0002). |
| RQ2: battery temperature in 5 min | XGBoost MAE 0.31 °C, 43% better than persistence, and 0.35 °C on unseen cars (H3, p = 0.0001). |
| RQ3: rest-of-trip energy | From the first 5 minutes, XGBoost is 20% better than the mean (MAE 4.75 SOC %/100 km, R² = 0.32; H4, p = 0.0001). |
| RQ4: thermal demand | Not started (3.4). |
| RQ5: battery health | Not started (3.5). |

All four tests stay significant after the Bonferroni correction.

## 4.4 Challenges and next steps

**Challenges**
- Sparse battery temperature: only about 6% of driving rows have a real pack-temperature reading, which limits RQ2 to 8,484 forecast rows. We use only real readings, to avoid leakage from interpolation.
- No battery current, motor RPM or odometer: energy is SOC % per 100 km (0.4% steps, so short trips are noisy), and distance comes from speed.
- Uneven data: 46 to 836 trips per car, and the CUPRAs were only logged in winter and spring, so car and season are mixed up. We shuffle within vehicle and also validate on unseen cars.

**Next steps**
- Tune hyperparameters with nested cross-validation.
- RQ2: try an LSTM on the 5-minute history if time allows, and compare pack max temperature with coolant outlet temperature as the target.
- RQ3: compare early windows of 3, 5 and 10 minutes.
- RQ1: decide whether to keep 4 regimes or merge the two urban ones (GMM result).
- RQ4: build the thermal-demand classifier (3.4).
- RQ5: decide whether to include the battery-health dataset (3.5).